# AFS Parse

Runs the full extraction pipeline on one or more filings in `COMMON.PDF_STAGING`.

**Steps:**
1. Select a filing from `PDF_STAGING` (status = `pending`)
2. Provide org details if this is a new organization
3. Run `pipeline.process_filing()` — identify → classify → extract → load
4. On success, status is set to `done`; on failure, `failed`

In [ ]:
import sys, json
sys.path.insert(0, '../src')

from afs import pipeline

In [ ]:
# Show pending filings
import pandas as pd

df = session.sql("""
    SELECT STAGING_ID, FILENAME, TOTAL_PAGES, STATUS, EXTRACTED_AT
      FROM AUDITED_FINANCIALS.COMMON.PDF_STAGING
     WHERE STATUS IN ('pending', 'failed')
     ORDER BY EXTRACTED_AT
""").to_pandas()

print(f'{len(df)} filing(s) ready to parse:')
df

In [ ]:
# ── CONFIGURE ─────────────────────────────────────────────────────────────────
# Set STAGING_ID to the value from the table above, or None to process all pending.
STAGING_ID  = None          # e.g. '3f8a...' or None for all pending

# For a NEW organization, provide org details. For known orgs (matched by EIN or name),
# leave as None — the pipeline will auto-match from ORG_REGISTRY.
ORG_HINT    = None          # e.g. {'org_code': 'ACME_HEALTH', 'legal_name': 'Acme Health System'}

REPARSE     = False         # Set True to re-extract even if filing_id already loaded
# ──────────────────────────────────────────────────────────────────────────────

In [ ]:
from snowflake.snowpark.functions import col

if STAGING_ID:
    staging_df = session.table('AUDITED_FINANCIALS.COMMON.PDF_STAGING').filter(
        col('STAGING_ID') == STAGING_ID
    )
else:
    staging_df = session.table('AUDITED_FINANCIALS.COMMON.PDF_STAGING').filter(
        col('STATUS').isin(['pending', 'failed'])
    )

rows = staging_df.select(
    col('STAGING_ID'),
    col('FILENAME'),
    col('FILING_ID'),
    col('TOTAL_PAGES'),
    col('PAGE_TEXTS').cast('STRING').alias('PAGE_TEXTS_STR'),
).order_by(col('EXTRACTED_AT')).collect()

print(f'Processing {len(rows)} filing(s)...')

In [ ]:
import logging
from snowflake.snowpark.functions import col, lit

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s %(message)s')

def _update_staging_status(session, staging_id, status):
    session.table('AUDITED_FINANCIALS.COMMON.PDF_STAGING').update(
        {'STATUS': status},
        col('STAGING_ID') == staging_id,
    )

for row in rows:
    staging_id  = row[0]
    filename    = row[1]
    filing_id   = row[2]
    total_pages = row[3]
    page_texts  = json.loads(row[4])

    print(f'\n=== {filename} ({total_pages} pages) ===')

    _update_staging_status(session, staging_id, 'processing')

    try:
        report = pipeline.process_filing(
            session,
            filename=filename,
            filing_id=filing_id,
            page_texts=page_texts,
            total_pages=total_pages,
            org_hint=ORG_HINT,
            reparse=REPARSE,
            staging_id=staging_id,
        )
        _update_staging_status(session, staging_id, 'done')
        print(json.dumps(report.get('stages', {}), indent=2, default=str))
        if report.get('skipped'):
            print(f'[skip] {report["skipped"]}')
        else:
            print('[ok]')
    except Exception as exc:
        _update_staging_status(session, staging_id, 'failed')
        print(f'[fail] {exc}')
        raise

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()